# Лабораторная работа 4
## Хеширование
**(20 баллов)**
Выполните самостоятельно следующие задания и оформите отчет.

In [ ]:
class HashTable:
    def __init__(self):
        self.size = 11
        self.slots = [None] * self.size
        self.data = [None] * self.size

    def put(self, key, data):
        hashvalue = self.hashfunction(key, len(self.slots))

        if self.slots[hashvalue] == None:
            self.slots[hashvalue] = key
            self.data[hashvalue] = data
        else:
            if self.slots[hashvalue] == key:
                self.data[hashvalue] = data  # replace
            else:
                nextslot = self.rehash(hashvalue, len(self.slots))
                while self.slots[nextslot] != None and \
                        self.slots[nextslot] != key:
                    nextslot = self.rehash(nextslot, len(self.slots))

                if self.slots[nextslot] == None:
                    self.slots[nextslot] = key
                    self.data[nextslot] = data
                else:
                    self.data[nextslot] = data  # replace

    def hashfunction(self, key, size):
        return key % size

    def rehash(self, oldhash, size):
        return (oldhash + 1) % size

    def get(self, key):
        startslot = self.hashfunction(key, len(self.slots))

        data = None
        stop = False
        found = False
        position = startslot
        while self.slots[position] != None and \
                not found and not stop:
            if self.slots[position] == key:
                found = True
                data = self.data[position]
            else:
                position = self.rehash(position, len(self.slots))
                if position == startslot:
                    stop = True
        return data

    def __getitem__(self, key):
        return self.get(key)

    def __setitem__(self, key, data):
        self.put(key, data)

№ 1

**(5 балла)**

Возьмите реализацию класса HashTable из лекционных материалов и выполните
следующие доработки:
1. Реализуйте квадратичное пробирование как технику повторного хеширования.
2. Реализуйте работу с функцией len (переопределите метод __len__).
3. Реализуйте работу оператора in (переопределите метод __contains__).
4. Переделайте метод put таким образом, чтобы таблица автоматически меняла размер,
когда загрузочный фактор становится больше значения 0.7. Размер должен
увеличиваться примерно в два раза до ближайшего подходящего простого числа.
5. Реализуйте работу оператора del (переопределите метод __delitem__) для удаления
элемента таблицы. Таблица должна автоматически менять размер, когда
загрузочный фактор становится меньше значения 0.2. Размер должен уменьшаться
примерно в два раза до ближайшего подходящего простого числа.
Все выполненные доработки должны быть протестированы

In [15]:
class HashTable:
    def __init__(self):
        self.size = 11
        self.slots = [None] * self.size
        self.data = [None] * self.size
        self.count = 0

    def is_prime(self, n):
        if n <= 1:
            return False
        if n <= 3:
            return True
        if n % 2 == 0 or n % 3 == 0:
            return False
        i = 5
        while i * i <= n:
            if n % i == 0 or n % (i + 2) == 0:
                return False
            i += 6
        return True
    
    def next_prime(self, n):
        while not self.is_prime(n):
            n += 1
        return n
    
    def prev_prime(self, n):
        n = max(2, n-1)
        while not self.is_prime(n):
            n -= 1
        return n
    
    def resize(self):
        old_slots = self.slots
        old_data = self.data
        new_size = self.next_prime(self.size * 2)
        self.size = new_size
        self.slots = [None] * self.size
        self.data = [None] * self.size
        self.count = 0

        for i in range(len(old_slots)):
            if old_slots[i] is not None and old_slots[i] != '<deleted>':
                self.put(old_slots[i], old_data[i])

    def put(self, key, data):
        if (self.count + 1) / self.size > 0.7:
            self.resize()
        
        hashvalue = self.hashfunction(key, len(self.slots))
        
        if self.slots[hashvalue] is None or self.slots[hashvalue] == '<deleted>':
            self.slots[hashvalue] = key
            self.data[hashvalue] = data
            self.count += 1
        else:
            if self.slots[hashvalue] == key:
                self.data[hashvalue] = data
            else:
                probe_count = 1
                nextslot = self.rehash(hashvalue, len(self.slots), probe_count)
                
                while (self.slots[nextslot] is not None and 
                       self.slots[nextslot] != key and 
                       self.slots[nextslot] != '<deleted>'):
                    probe_count += 1
                    nextslot = self.rehash(hashvalue, len(self.slots), probe_count)
                    
                    if probe_count > self.size:
                        raise Exception("Хеш-таблица переполнена")

                if self.slots[nextslot] is None or self.slots[nextslot] == '<deleted>':
                    self.slots[nextslot] = key
                    self.data[nextslot] = data
                    self.count += 1
                else:
                    self.data[nextslot] = data

    def hashfunction(self, key, size):
        return key % size

    def rehash(self, oldhash, size, probe_count):
        return (oldhash + probe_count * probe_count) % size

    def get(self, key):
        startslot = self.hashfunction(key, len(self.slots))

        data = None
        stop = False
        found = False
        position = startslot
        probe_count = 0
        
        while (self.slots[position] is not None and 
               not found and not stop):
            
            if self.slots[position] == key:
                found = True
                data = self.data[position]
            else:
                probe_count += 1
                position = self.rehash(startslot, len(self.slots), probe_count)
                
                if position == startslot:
                    stop = True

                elif probe_count >= self.size:
                    stop = True

        return data

    def __getitem__(self, key):
        return self.get(key)

    def __setitem__(self, key, data):
        self.put(key, data)

    def __len__(self):
        return self.count
    
    def __contains__(self, key):
        startslot = self.hashfunction(key, len(self.slots))

        position = startslot
        probe_count = 0
        stop = False
        found = False

        while self.slots[position] is not None and not found and not stop:
            if self.slots[position] == key:
                found = True
            else:
                probe_count += 1
                position = self.rehash(startslot, len(self.slots), probe_count)

                if position == startslot or probe_count >= self.size:
                    stop = True

        return found
    
    def __delitem__(self, key):
        startslot = self.hashfunction(key, self.size)
        position = startslot
        probe_count = 0
        found = False
        stop = False

        while self.slots[position] is not None and not found and not stop:
            if self.slots[position] == key:
                found = True
            else:
                probe_count += 1
                position = self.rehash(startslot, self.size, probe_count)
                if position == startslot or probe_count >= self.size:
                    stop = True

        if found:
            self.slots[position] = '<deleted>'
            self.data[position] = None
            self.count -= 1

            if self.count / self.size < 0.2 and self.size > 11:
                new_size = self.prev_prime(max(11, self.size // 2))
                self.resize(new_size)
        else:
            raise KeyError(f"Ключ {key} не найден для удаления")

    def display(self):
        print("Slots:", self.slots)
        print("Data: ", self.data)

In [17]:
hash_table = HashTable()

hash_table[54] = 54
hash_table[26] = 26
hash_table[93] = 93
hash_table[40] = 40

print("После добавления первых значений:")
hash_table.display()
print(f"Текущий размер: {len(hash_table)}")

hash_table[15] = 15
print("\nПосле добавления 15 (коллизия с 26):")
hash_table.display()
print(f"Текущий размер: {len(hash_table)}")

print(f"\nПоиск ключа 15: {hash_table[15]}")
print(f"Поиск несуществующего ключа 29: {hash_table[29]}")

hash_table[15] = 150
print("\nПосле обновления значения по ключу 15:")
print(f"Поиск ключа 15: {hash_table[15]}")
print(f"Текущий размер: {len(hash_table)}")

print(f"\nКлюч 15 в таблице? {'Да' if 15 in hash_table else 'Нет'}")
print(f"Ключ 29 в таблице? {'Да' if 29 in hash_table else 'Нет'}")
print(f"Ключ 54 в таблице? {'Да' if 54 in hash_table else 'Нет'}")

for key in [1, 2, 3, 4, 5, 6]:
    hash_table[key] = key
print("\nПосле добавления дополнительных ключей:")
hash_table.display()
print(f"Новый размер таблицы: {hash_table.size}")
print(f"Текущий размер: {len(hash_table)}")

del hash_table[1]
del hash_table[2]
del hash_table[3]
del hash_table[4]
del hash_table[5]
del hash_table[6]

print("\nПосле удаления некоторых ключей:")
hash_table.display()
print(f"Новый размер таблицы: {hash_table.size}")
print(f"Текущий размер: {len(hash_table)}")

После добавления первых значений:
Slots: [None, None, None, None, 26, 93, None, 40, None, None, 54]
Data:  [None, None, None, None, 26, 93, None, 40, None, None, 54]
Текущий размер: 4

После добавления 15 (коллизия с 26):
Slots: [None, None, None, None, 26, 93, None, 40, 15, None, 54]
Data:  [None, None, None, None, 26, 93, None, 40, 15, None, 54]
Текущий размер: 5

Поиск ключа 15: 15
Поиск несуществующего ключа 29: None

После обновления значения по ключу 15:
Поиск ключа 15: 150
Текущий размер: 5

Ключ 15 в таблице? Да
Ключ 29 в таблице? Нет
Ключ 54 в таблице? Да

После добавления дополнительных ключей:
Slots: [None, 1, 2, 26, 3, 93, 5, 6, 54, None, None, None, None, 4, None, 15, None, 40, None, None, None, None, None]
Data:  [None, 1, 2, 26, 3, 93, 5, 6, 54, None, None, None, None, 4, None, 150, None, 40, None, None, None, None, None]
Новый размер таблицы: 23
Текущий размер: 11

После удаления некоторых ключей:
Slots: [None, '<deleted>', '<deleted>', 26, '<deleted>', 93, '<deleted>',

№ 2

**(5 балла)**

Возьмите реализацию класса HashTable из лекционных материалов и выполните
следующие доработки:
1. Переделайте существующие методы так, чтобы разрешение коллизий происходило
не при помощи концепции открытой адресации, а методом цепочек. Для этого в
каждом слоте храните связный список, реализованный классом UnorderedList из
лабораторной работы 3.
2. Реализуйте работу с функцией len (переопределите метод __len__).
3. Реализуйте работу оператора in (переопределите метод __contains__).
4. Переделайте метод put таким образом, чтобы таблица автоматически меняла размер,
когда загрузочный фактор становится больше значения 0.7. Размер должен
увеличиваться примерно в два раза до ближайшего подходящего простого числа.
5. Реализуйте работу оператора del (переопределите метод __delitem__) для удаления
элемента таблицы. Таблица должна автоматически менять размер, когда
загрузочный фактор становится меньше значения 0.2. Размер должен уменьшаться
примерно в два раза до ближайшего подходящего простого числа.
Все выполненные доработки должны быть протестированы.


In [35]:
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

class UnorderedList:
    def __init__(self):
        self.head = None

    def add(self, data):
        new_node = Node(data)
        new_node.next = self.head
        self.head = new_node

    def search(self, key):
        current = self.head
        while current is not None:
            if current.data[0] == key:
                return current
            current = current.next
        return None
    
    def __iter__(self):
        current = self.head
        while current is not None:
            yield current.data
            current = current.next

class HashTable:
    def __init__(self):
        self.size = 11
        self.slots = [UnorderedList() for _ in range(self.size)]
        self.count = 0

    def is_prime(self, n):
        if n <= 1:
            return False
        if n <= 3:
            return True
        if n % 2 == 0 or n % 3 == 0:
            return False
        i = 5
        while i * i <= n:
            if n % i == 0 or n % (i + 2) == 0:
                return False
            i += 6
        return True

    def next_prime(self, n):
        while not self.is_prime(n):
            n += 1
        return n

    def resize(self, new_size):
        old_slots = self.slots
        self.size = new_size
        self.slots = [UnorderedList() for _ in range(self.size)]
        self.count = 0
        for chain in old_slots:
            for key, data in chain:
                self.put(key, data)

    def put(self, key, data):
        if (self.count + 1) / self.size > 0.7:
            new_size = self.next_prime(self.size * 2)
            self.resize(new_size)

        slot = self.hashfunction(key, self.size)
        chain = self.slots[slot]

        node = chain.search(key)
        if node is None:
            chain.add((key, data))
            self.count += 1
        else:
            node.data = (key, data)

    def hashfunction(self, key, size):
        return key % size

    def get(self, key):
        slot = self.hashfunction(key, self.size)
        chain = self.slots[slot]
        node = chain.search(key)
        if node is None:
            return None
        return node.data[1]

    def __getitem__(self, key):
        return self.get(key)

    def __setitem__(self, key, data):
        self.put(key, data)

    def __len__(self):
        count = 0
        for chain in self.slots:
            for _ in chain:
                count += 1
        return count
    
    def __contains__(self, key):
        slot = self.hashfunction(key, self.size)
        chain = self.slots[slot]
        node = chain.search(key)
        return node is not None
    
    def __delitem__(self, key):
        slot = self.hashfunction(key, self.size)
        chain = self.slots[slot]

        current = chain.head
        previous = None
        found = False

        while current is not None and not found:
            if current.data[0] == key:
                found = True
            else:
                previous = current
                current = current.next

        if not found:
            raise KeyError(f"Ключ {key} не найден для удаления")

        if previous is None:
            chain.head = current.next
        else:
            previous.next = current.next

        self.count -= 1

        if self.count / self.size < 0.2 and self.size > 11:
            new_size = self.next_prime(max(11, self.size // 2))
            self.resize(new_size)

    def display(self):
        print("Hash Table Contents:")
        for index, chain in enumerate(self.slots):
            items = list(chain)
            if items:
                print(f"Slot {index}: {items}")
            else:
                print(f"Slot {index}: []")


In [ ]:
hash_table = HashTable()

hash_table[54] = 54
hash_table[26] = 26
hash_table[93] = 93
hash_table[40] = 40

print("После добавления первых значений:")
hash_table.display()
print(f"Текущий размер: {len(hash_table)}")

hash_table[15] = 15
print("\nПосле добавления 15 (коллизия с 26):")
hash_table.display()
print(f"Текущий размер: {len(hash_table)}")

print(f"\nПоиск ключа 15: {hash_table[15]}")
print(f"Поиск несуществующего ключа 29: {hash_table[29]}")

hash_table[15] = 150
print("\nПосле обновления значения по ключу 15:")
print(f"Поиск ключа 15: {hash_table[15]}")
print(f"Текущий размер: {len(hash_table)}")

print(f"\nКлюч 15 в таблице? {'Да' if 15 in hash_table else 'Нет'}")
print(f"Ключ 29 в таблице? {'Да' if 29 in hash_table else 'Нет'}")
print(f"Ключ 54 в таблице? {'Да' if 54 in hash_table else 'Нет'}")


for key in [1, 2, 3, 4, 5, 6]:
    hash_table[key] = key
print("\nПосле добавления дополнительных ключей:")
hash_table.display()
print(f"Новый размер таблицы: {hash_table.size}")
print(f"Текущий размер: {len(hash_table)}")

del hash_table[1]
del hash_table[2]
del hash_table[3]
del hash_table[4]
del hash_table[5]
del hash_table[6]

print("\nПосле удаления некоторых ключей:")
hash_table.display()
print(f"Новый размер таблицы: {hash_table.size}")
print(f"Текущий размер: {len(hash_table)}")


После добавления первых значений:
Hash Table Contents:
Slot 0: []
Slot 1: []
Slot 2: []
Slot 3: []
Slot 4: [(26, 26)]
Slot 5: [(93, 93)]
Slot 6: []
Slot 7: [(40, 40)]
Slot 8: []
Slot 9: []
Slot 10: [(54, 54)]
Текущий размер: 4

После добавления 15 (коллизия с 26):
Hash Table Contents:
Slot 0: []
Slot 1: []
Slot 2: []
Slot 3: []
Slot 4: [(15, 15), (26, 26)]
Slot 5: [(93, 93)]
Slot 6: []
Slot 7: [(40, 40)]
Slot 8: []
Slot 9: []
Slot 10: [(54, 54)]
Текущий размер: 5

Поиск ключа 15: 15
Поиск несуществующего ключа 29: None

После обновления значения по ключу 15:
Поиск ключа 15: 150
Текущий размер: 5

Ключ 15 в таблице? Да
Ключ 29 в таблице? Нет
Ключ 54 в таблице? Да

После добавления дополнительных ключей:
Hash Table Contents:
Slot 0: []
Slot 1: [(93, 93), (1, 1)]
Slot 2: [(2, 2)]
Slot 3: [(3, 3), (26, 26)]
Slot 4: [(4, 4)]
Slot 5: [(5, 5)]
Slot 6: [(6, 6)]
Slot 7: []
Slot 8: [(54, 54)]
Slot 9: []
Slot 10: []
Slot 11: []
Slot 12: []
Slot 13: []
Slot 14: []
Slot 15: [(15, 150)]
Slot 16: []


№ 3

**(2 балла)**

Переделайте класс HashTable, чтобы в качестве ключей можно было использовать строки.

In [37]:
class HashTable:
    def __init__(self):
        self.size = 11
        self.slots = [UnorderedList() for _ in range(self.size)]
        self.count = 0

    def is_prime(self, n):
        if n <= 1:
            return False
        if n <= 3:
            return True
        if n % 2 == 0 or n % 3 == 0:
            return False
        i = 5
        while i * i <= n:
            if n % i == 0 or n % (i + 2) == 0:
                return False
            i += 6
        return True

    def next_prime(self, n):
        while not self.is_prime(n):
            n += 1
        return n

    def resize(self, new_size):
        old_slots = self.slots
        self.size = new_size
        self.slots = [UnorderedList() for _ in range(self.size)]
        self.count = 0
        for chain in old_slots:
            for key, data in chain:
                self.put(key, data)

    def put(self, key, data):
        if (self.count + 1) / self.size > 0.7:
            new_size = self.next_prime(self.size * 2)
            self.resize(new_size)

        slot = self.hashfunction(key, self.size)
        chain = self.slots[slot]

        node = chain.search(key)
        if node is None:
            chain.add((key, data))
            self.count += 1
        else:
            node.data = (key, data)

    def hashfunction(self, key, size):
        return hash(key) % size

    def get(self, key):
        slot = self.hashfunction(key, self.size)
        chain = self.slots[slot]
        node = chain.search(key)
        if node is None:
            return None
        return node.data[1]

    def __getitem__(self, key):
        return self.get(key)

    def __setitem__(self, key, data):
        self.put(key, data)

    def __len__(self):
        return self.count

    def __contains__(self, key):
        slot = self.hashfunction(key, self.size)
        chain = self.slots[slot]
        node = chain.search(key)
        return node is not None

    def __delitem__(self, key):
        slot = self.hashfunction(key, self.size)
        chain = self.slots[slot]

        current = chain.head
        previous = None
        found = False

        while current is not None and not found:
            if current.data[0] == key:
                found = True
            else:
                previous = current
                current = current.next

        if not found:
            raise KeyError(f"Ключ {key} не найден для удаления")

        if previous is None:
            chain.head = current.next
        else:
            previous.next = current.next

        self.count -= 1

        if self.count / self.size < 0.2 and self.size > 11:
            new_size = self.next_prime(max(11, self.size // 2))
            self.resize(new_size)

    def display(self):
        print("Hash Table Contents:")
        for index, chain in enumerate(self.slots):
            items = list(chain)
            if items:
                print(f"Slot {index}: {items}")
            else:
                print(f"Slot {index}: []")


In [38]:
hash_table = HashTable()

hash_table[54] = "value54"
hash_table[26] = "value26"
hash_table[93] = "value93"
hash_table[40] = "value40"

print("После добавления числовых ключей:")
hash_table.display()
print(f"Текущий размер: {len(hash_table)}")

hash_table["apple"] = "fruit"
hash_table["carrot"] = "vegetable"
hash_table["banana"] = "fruit"

print("\nПосле добавления строковых ключей:")
hash_table.display()
print(f"Текущий размер: {len(hash_table)}")

print(f"\nПоиск ключа 'apple': {hash_table['apple']}")
print(f"Поиск ключа 26: {hash_table[26]}")
print(f"Поиск отсутствующего ключа 'orange': {hash_table['orange']}")

hash_table["apple"] = "green fruit"
print("\nПосле обновления значения по ключу 'apple':")
print(f"Поиск ключа 'apple': {hash_table['apple']}")

print(f"\nКлюч 'carrot' в таблице? {'Да' if 'carrot' in hash_table else 'Нет'}")
print(f"Ключ 'orange' в таблице? {'Да' if 'orange' in hash_table else 'Нет'}")

После добавления числовых ключей:
Hash Table Contents:
Slot 0: []
Slot 1: []
Slot 2: []
Slot 3: []
Slot 4: [(26, 'value26')]
Slot 5: [(93, 'value93')]
Slot 6: []
Slot 7: [(40, 'value40')]
Slot 8: []
Slot 9: []
Slot 10: [(54, 'value54')]
Текущий размер: 4

После добавления строковых ключей:
Hash Table Contents:
Slot 0: []
Slot 1: []
Slot 2: []
Slot 3: [('banana', 'fruit'), ('apple', 'fruit')]
Slot 4: [(26, 'value26')]
Slot 5: [(93, 'value93')]
Slot 6: []
Slot 7: [(40, 'value40')]
Slot 8: [('carrot', 'vegetable')]
Slot 9: []
Slot 10: [(54, 'value54')]
Текущий размер: 7

Поиск ключа 'apple': fruit
Поиск ключа 26: value26
Поиск отсутствующего ключа 'orange': None

После обновления значения по ключу 'apple':
Поиск ключа 'apple': green fruit

Ключ 'carrot' в таблице? Да
Ключ 'orange' в таблице? Нет
